## Geração e Divisão dos Dados

In [ ]:
import numpy as np
import os
from datasets import UORED

domains = {
    '1': ('H-1-0', 'I-1-1', 'O-6-1', 'B-11-1'),
    '2': ('H-2-0', 'I-1-2', 'O-6-2', 'B-11-2'),
    '3': ('H-3-0', 'I-2-1', 'O-7-1', 'B-12-1'),
    '4': ('H-4-0', 'I-2-2', 'O-7-2', 'B-12-2'),
    '5': ('H-5-0', 'I-3-1', 'O-8-1', 'B-13-1'),
    '6': ('H-6-0', 'I-3-2', 'O-8-2', 'B-13-2'),
    '7': ('H-7-0', 'I-4-1', 'O-9-1', 'B-14-1'),
    '8': ('H-8-0', 'I-4-2', 'O-9-2', 'B-14-2'),
    '9': ('H-9-0', 'I-5-1', 'O-10-1', 'B-15-1'),
    '10': ('H-10-0', 'I-5-2', 'O-10-2', 'B-15-2'),
}


def create_and_save_segments(filepath, output_path, max_allowed_sample_size=484_400, segment_size=2048, transform=None):
    dataset = UORED()
    signal = dataset.load_file(filepath)[0]
    if transform is not None:
        signal = transform(signal)
    max_allowed_sample_size = len(signal)
    num_segments = max_allowed_sample_size//segment_size
    for i in range(num_segments):
        segment = signal[i*(segment_size):(i+1)*segment_size]
        np.save(f"{output_path}_{i}", segment)

classes = ("B", "I", "N", "O")
train_map = {
    'N': ['H_2_0', 'H_3_0', 'H_4_0', 'H_5_0'],
    'I': ['I_2_2', 'I_3_2', 'I_4_2', 'I_5_2'],
    'B': ['B_12_1', 'B_13_2', 'B_14_2', 'B_15_2'],
    'O': ['O_7_2', 'O_8_2', 'O_9_2', 'O_10_2']
}
test_map = {
    'N': ['H_1_0'],
    'I': ['I_1_2'],
    'B': ['B_11_2'],
    'O': ['O_6_2'],
}

maps = {
    'train': train_map,
    'test': test_map,
    'val': test_map
}

mode = 'val'
for nr in [1]: #range(1, 10):
    r = f'round_{nr}'
    output_dir = f"data/processed/uored_512_12/{r}/{mode}/"

    # Cria a arvore de diretórios
    for label in classes:
        os.makedirs(f"{output_dir}/{label}", exist_ok=True)
    print("Directory tree created!")

    mapping = maps[mode]
    for label in mapping:
        basenames = mapping[label]
        for basename in basenames:
            output_path = f"{output_dir}/{label}/{basename}"            
            create_and_save_segments(f"data/raw/uored/{basename}.mat", output_path, segment_size=512, transform=resample_signal)
    print("Finish!")

2